Imports

In [32]:
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
import torch.nn.functional as F
import Bio.PDB
from Bio.PDB import PDBList



Loading Dataset

In [33]:
df = pd.read_csv("Datasets/Fluorescent-Protein-Database.csv")


making PDB filenames

In [38]:
df["PDB_filename"] = df["PDB_code"].str.lower().apply(
    lambda x: f"pdbs/pdb{x}.ent"
)
df["PDB_code"] = df["PDB code"]
#PDB_code more digestible for python as python despises spaces in strings


Creating Contact Map Matrices using Biopython

In [ ]:
def residue_dist(residue_one, residue_two) :
    """Returns the C-alpha distance between two residues"""
    diff_vector  = residue_one["CA"].coord - residue_two["CA"].coord
    return np.sqrt(np.sum(diff_vector * diff_vector))

def dist_matrix(chain_one, chain_two) :
    """Returns a matrix of C-alpha distances between two chains"""
    answer = np.zeros((len(chain_one), len(chain_two)), float)
    for row, residue_one in enumerate(chain_one) :
        for col, residue_two in enumerate(chain_two) :
            answer[row, col] = residue_dist(residue_one, residue_two)
    return answer

def get_distance_matrices(pdb_code, pdb_filename):
    parser = Bio.PDB.PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_code, pdb_filename)
    model = structure[0]

    chain = next(model.get_chains())

    residues = [r for r in chain if "CA" in r]

    return dist_matrix(residues, residues)

pdbl = PDBList()

for code in df["PDB_code"]:
    pdbl.retrieve_pdb_file(code, pdir="pdbs", file_format="pdb")



df["dist_matrix"] = df.apply(
    lambda row: get_distance_matrices(row["PDB_code"], row["PDB_filename"]),
    axis=1
)


Structure exists: 'pdbs\pdb3kct.ent' 
Structure exists: 'pdbs\pdb7oin.ent' 
Structure exists: 'pdbs\pdb8arm.ent' 
Structure exists: 'pdbs\pdb3nt3.ent' 
Structure exists: 'pdbs\pdb3u0l.ent' 
Structure exists: 'pdbs\pdb7ry2.ent' 
Structure exists: 'pdbs\pdb6u1a.ent' 
Structure exists: 'pdbs\pdb3wck.ent' 
Structure exists: 'pdbs\pdb3gb3.ent' 
Structure exists: 'pdbs\pdb1uis.ent' 
Structure exists: 'pdbs\pdb3e5t.ent' 
Structure exists: 'pdbs\pdb8xc6.ent' 
Structure exists: 'pdbs\pdb3ir8.ent' 
Structure exists: 'pdbs\pdb4nws.ent' 
Structure exists: 'pdbs\pdb7qgk.ent' 
Structure exists: 'pdbs\pdb3svn.ent' 
Structure exists: 'pdbs\pdb3nez.ent' 
Structure exists: 'pdbs\pdb3pj5.ent' 
Structure exists: 'pdbs\pdb3bxa.ent' 
Structure exists: 'pdbs\pdb3ned.ent' 
Structure exists: 'pdbs\pdb2qlg.ent' 
Structure exists: 'pdbs\pdb4edo.ent' 
Structure exists: 'pdbs\pdb3ip2.ent' 
Structure exists: 'pdbs\pdb4oqw.ent' 
Structure exists: 'pdbs\pdb4eds.ent' 
Structure exists: 'pdbs\pdb4kgf.ent' 
Structure ex

The obtained nxn matrices aren't of shape suitable for concatenation with other descriptors and are (somewhat brutally) flattened into vectors:

In [49]:
df["flattened"] = df["dist_matrix"].apply(lambda x: x.flatten())